# 🎙️🎭 Server GABUNGAN VoxCPM2 + Talking Avatar (PippitLokal)

Satu Colab, satu URL. Endpoint `/tts` (voice clone) **dan** `/avatar` (video talking avatar) jalan bersama.

**Cara pakai:** Runtime → Change runtime type → **T4 GPU** → Save, lalu **Runtime → Run all**.
Tunggu sampai muncul **URL SERVER** (…trycloudflare.com). Salin URL itu ke PippitLokal:
- kolom *URL server VoxCPM* (voice clone), dan
- setting `avatarColabUrl` (pakai URL yang **sama**).


### 1) Install VoxCPM + engine avatar (SadTalker + LivePortrait)
Sekali per sesi. Bisa ~5-10 menit (download model).

In [ ]:
# Install voice + clone repo + setup engine avatar
!pip -q install voxcpm soundfile
!git clone -b feat/talking-avatar https://github.com/dionisius95/yt-short.git /content/yt-short 2>/dev/null || (cd /content/yt-short && git fetch origin feat/talking-avatar && git reset --hard origin/feat/talking-avatar)
!bash /content/yt-short/colab/setup_colab.sh


### 2) Set environment engine avatar (T4-friendly)

In [ ]:
import os
os.environ['SADTALKER_DIR']    = '/content/SadTalker'
os.environ['LIVEPORTRAIT_DIR'] = '/content/LivePortrait'
os.environ['IDLE_DRIVING']     = '/content/assets/idle_driving.mp4'
os.environ['AVATAR_MAX_SIDE']  = '256'   # avatar tampil ~25% layar -> 256 cukup & ~3-4x lebih cepat (hindari timeout Segmen C talk)
os.environ['AVATAR_FP16']      = '1'
os.environ['AVATAR_FPS']       = '25'
os.environ['AVATAR_WORKDIR']   = '/content/avatar_work'

### 3) Tulis server gabungan (otomatis, tidak perlu upload apa pun)

In [ ]:
%%writefile /content/pippit_server.py
# -*- coding: utf-8 -*-
"""
pippit_server.py - server GABUNGAN untuk PippitLokal.

Satu port, satu URL cloudflared:
  GET  /health         -> 'ok' (200 HANYA setelah model VoxCPM termuat)
  GET  /avatar/health  -> JSON status engine avatar (talk/idle/device)
  POST /tts  (atau /clone) -> audio/wav  (voice clone VoxCPM2)
  POST /avatar (atau /lipsync) -> 202 JSON { job_id, status }  (ASYNC)
  GET  /avatar/result/<job_id> -> 202 (proses) | video/mp4 (selesai) | 5xx JSON (gagal)

Avatar render dijalankan di thread background supaya tiap request pendek dan
tidak pernah kena batas ~100s Cloudflare quick tunnel (HTTP 524). Fungsi render
diimpor langsung dari repo yt-short (colab/avatar_server.py).
"""
import os, io, sys, json, base64, tempfile, traceback, subprocess, threading, uuid, time
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import soundfile as sf
from voxcpm import VoxCPM

# ---- Impor fungsi render avatar (Flask-independent) dari repo ----
sys.path.insert(0, "/content/yt-short/colab")
from avatar_server import (
    _prep_image, _render_sadtalker, _render_liveportrait_idle,
    _make_silent_wav, _loop_to_duration, _sadtalker_available, _liveportrait_available,
    WORKDIR, DEFAULT_FPS,
)
try:
    from avatar_server import remove_background_video
except Exception:
    remove_background_video = None

MODEL_ID = os.environ.get("VOXCPM_MODEL", "openbmb/VoxCPM2")
DEVICE   = os.environ.get("VOXCPM_DEVICE", "auto")
PORT     = int(os.environ.get("VOXCPM_PORT", "8081"))

print(f"[VoxCPM] Memuat model {MODEL_ID} (device={DEVICE}) ...", flush=True)
try:
    MODEL = VoxCPM.from_pretrained(MODEL_ID, load_denoiser=False, device=DEVICE)
    try:
        MODEL.generate(text="Warmup test audio")
    except Exception:
        pass
    print("[VoxCPM] Model siap 100% untuk voice cloning!", flush=True)
except Exception:
    print("[VoxCPM] GAGAL memuat model:", flush=True)
    traceback.print_exc()
    sys.exit(1)


def _to_wav(wav):
    buf = io.BytesIO()
    sf.write(buf, wav, MODEL.tts_model.sample_rate, format="WAV")
    return buf.getvalue()


def _synth(text, prompt_text, ref_wav_path):
    if ref_wav_path and os.path.exists(ref_wav_path) and os.path.getsize(ref_wav_path) > 100:
        print(f"[VoxCPM] Voice cloning dgn sampel ({os.path.getsize(ref_wav_path)} bytes)...", flush=True)
        try:
            wav = MODEL.generate(text=text, reference_wav_path=ref_wav_path)
            print("[VoxCPM] SUKSES: suara dikloning persis sampel!", flush=True)
            return _to_wav(wav)
        except Exception as e1:
            print(f"[VoxCPM] reference_wav_path gagal ({e1}), coba prompt_wav_path...", flush=True)
            wav = MODEL.generate(text=text, prompt_wav_path=ref_wav_path)
            print("[VoxCPM] SUKSES: voice cloning via prompt_wav_path!", flush=True)
            return _to_wav(wav)
    else:
        raise ValueError("File sampel suara kosong atau tidak ditemukan oleh server VoxCPM.")


def _render_static_idle(img_path, out_dir, fps, duration):
    """Idle TENANG tanpa gerak wajah: potret statis + micro-zoom sangat halus
    supaya terbaca sebagai shot 'hidup' (bukan foto beku) TAPI tidak ada
    mulut komat-kamit / alis liar. Deterministik, tanpa ML, selalu cepat."""
    from pathlib import Path as _P
    out_dir = _P(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / "idle_static.mp4"
    dur = max(0.5, float(duration))
    nframes = max(1, int(round(dur * int(fps))))
    vf = (
        "scale=512:512:force_original_aspect_ratio=increase,crop=512:512,"
        "zoompan=z='min(1.0+0.0006*on,1.06)':d=%d:s=256x256:fps=%d,"
        "format=yuv420p" % (nframes, int(fps))
    )
    subprocess.run([
        "ffmpeg", "-y", "-loop", "1", "-i", str(img_path),
        "-t", "%.3f" % dur, "-r", str(int(fps)), "-vf", vf,
        "-c:v", "libx264", "-crf", "20", "-pix_fmt", "yuv420p", str(out),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)
    return out


def _render_avatar_to_file(data):
    """Render mp4 avatar dan balikan PATH file (tidak dihapus). Reuse avatar_server.py."""
    image_b64 = data.get("image_b64")
    audio_b64 = data.get("audio_b64")
    mode = (data.get("mode") or "talk").lower()
    duration = float(data.get("duration") or 4.0)
    fps = int(data.get("fps") or DEFAULT_FPS)

    if not image_b64:
        raise ValueError("image_b64 is required")
    if mode == "talk" and not audio_b64:
        raise ValueError("audio_b64 is required for mode talk")
    if not _sadtalker_available():
        raise RuntimeError("SadTalker belum terpasang di host ini")

    job = Path(tempfile.mkdtemp(prefix="job_", dir=str(WORKDIR)))
    img_path = _prep_image(base64.b64decode(image_b64), job / "src.png")
    out_dir = job / "out"

    if mode == "idle":
        try:
            if _liveportrait_available():
                vid = _render_liveportrait_idle(img_path, out_dir, duration, fps)
            else:
                raise RuntimeError("LivePortrait unavailable")
        except Exception as e:
            print("[avatar] idle -> STATIS tenang (LivePortrait tak tersedia:", e, ")", flush=True)
            # PENTING: JANGAN pakai SadTalker utk idle. SadTalker selalu
            # menganimasikan wajah walau audio senyap -> mulut komat-kamit &
            # alis bergerak liar (keluhan user). Idle tenang = potret statis
            # + micro-zoom halus. Blink/senyum natural hanya lewat cabang
            # LivePortrait di atas (butuh idle_driving.mp4).
            vid = _render_static_idle(img_path, out_dir, fps, float(duration))
    else:
        aud_path = job / "drive.wav"
        aud_path.write_bytes(base64.b64decode(audio_b64))
        vid = _render_sadtalker(img_path, aud_path, out_dir, fps)

    final = job / "avatar.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(vid),
        "-r", str(fps), "-pix_fmt", "yuv420p",
        "-movflags", "+faststart", "-c:v", "libx264", "-crf", "20",
        str(final),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)

    # Hapus background (opsional). Toggle OFF -> lewati total, pipeline normal.
    if bool(data.get("remove_bg")) and remove_background_video is not None:
        try:
            matted = remove_background_video(str(final), str(out_dir), int(fps))
            if matted and os.path.exists(str(matted)) and str(matted) != str(final):
                print("[avatar] background dihapus -> %s" % matted, flush=True)
                return Path(matted)
            print("[avatar] remove_bg tak menghasilkan webm, pakai mp4 biasa", flush=True)
        except Exception as e:
            print("[avatar] remove_bg gagal, fallback mp4:", e, flush=True)
    return final


# ---- Registry job async untuk avatar ----
_JOBS = {}
_JOBS_LOCK = threading.Lock()
_RENDER_LOCK = threading.Lock()  # serialisasi GPU (VoxCPM + SadTalker berbagi 1 T4)


def _run_avatar_job(job_id, data):
    t0 = time.time()
    with _JOBS_LOCK:
        if job_id in _JOBS:
            _JOBS[job_id]["status"] = "running"
    try:
        with _RENDER_LOCK:
            final = _render_avatar_to_file(data)
        with _JOBS_LOCK:
            _JOBS[job_id].update(status="done", path=str(final))
        print("[avatar] job %s (%s) selesai %.1fs -> %s" % (job_id, data.get("mode"), time.time() - t0, final), flush=True)
    except Exception as e:
        traceback.print_exc()
        with _JOBS_LOCK:
            _JOBS[job_id].update(status="error", error=str(e))


class H(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def _send_json(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        p = self.path.rstrip("/")
        # Hasil job avatar (async polling)
        if p.startswith("/avatar/result/"):
            job_id = p[len("/avatar/result/"):]
            with _JOBS_LOCK:
                job = dict(_JOBS.get(job_id) or {})
            if not job:
                self._send_json(404, {"status": "error", "error": "unknown job_id"})
                return
            st = job.get("status")
            if st in ("pending", "running"):
                self._send_json(202, {"status": st})
                return
            if st == "error":
                self._send_json(500, {"status": "error", "error": job.get("error") or "render failed"})
                return
            path = job.get("path")
            if not path or not os.path.exists(path):
                self._send_json(500, {"status": "error", "error": "result file missing"})
                return
            blob = Path(path).read_bytes()
            mime = "video/webm" if str(path).lower().endswith(".webm") else "video/mp4"
            self.send_response(200)
            self.send_header("Content-Type", mime)
            self.send_header("Content-Length", str(len(blob)))
            self.end_headers()
            self.wfile.write(blob)
            return
        if p == "/avatar/health":
            body = json.dumps({
                "status": "ok",
                "talk": _sadtalker_available(),
                "idle": _liveportrait_available() or _sadtalker_available(),
            }).encode()
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(body)
            return
        if p in ["/health", "", "/tts", "/clone", "/avatar", "/lipsync"]:
            self.send_response(200); self.end_headers(); self.wfile.write(b"ok")
        else:
            self.send_response(404); self.end_headers()

    def do_POST(self):
        p = self.path.rstrip("/")
        try:
            n = int(self.headers.get("Content-Length", 0))
            data = json.loads(self.rfile.read(n) or b"{}")
        except Exception as e:
            self._send_json(400, {"error": f"bad json: {e}"})
            return

        # ---- Avatar ASYNC: enqueue job -> balikan 202 job_id segera ----
        if p in ["/avatar", "/lipsync"]:
            try:
                if not data.get("image_b64"):
                    raise ValueError("image_b64 is required")
                if (data.get("mode") or "talk").lower() == "talk" and not data.get("audio_b64"):
                    raise ValueError("audio_b64 is required for mode talk")
                job_id = uuid.uuid4().hex
                with _JOBS_LOCK:
                    _JOBS[job_id] = {"status": "pending", "path": None, "error": None}
                threading.Thread(target=_run_avatar_job, args=(job_id, data), daemon=True).start()
                print("[avatar] queued job", job_id, "mode", (data.get("mode") or "talk"), flush=True)
                self._send_json(202, {"job_id": job_id, "status": "pending"})
            except Exception as e:
                traceback.print_exc()
                self._send_json(400, {"error": str(e)})
            return

        # ---- Voice clone (audio/wav) - tetap sinkron ----
        if p in ["/tts", "/clone", ""]:
            try:
                text = data.get("text", "")
                prompt_text = data.get("prompt_text", "")
                ref_b64 = data.get("ref_audio_b64") or data.get("speaker_wav_b64")
                ref_path = None
                if ref_b64:
                    raw_tmp = tempfile.NamedTemporaryFile(suffix=".raw", delete=False)
                    raw_tmp.write(base64.b64decode(ref_b64))
                    raw_tmp.close()
                    clean_wav = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
                    subprocess.run(["ffmpeg", "-y", "-i", raw_tmp.name, "-ar", "16000", "-ac", "1", clean_wav],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    try: os.unlink(raw_tmp.name)
                    except Exception: pass
                    if os.path.exists(clean_wav) and os.path.getsize(clean_wav) > 100:
                        ref_path = clean_wav
                audio = _synth(text, prompt_text, ref_path)
                if ref_path and os.path.exists(ref_path):
                    try: os.unlink(ref_path)
                    except Exception: pass
                self.send_response(200)
                self.send_header("Content-Type", "audio/wav")
                self.send_header("Content-Length", str(len(audio)))
                self.end_headers()
                self.wfile.write(audio)
            except Exception as e:
                traceback.print_exc()
                msg = json.dumps({"error": str(e)}).encode()
                self.send_response(500)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(msg)
            return

        self.send_response(404); self.end_headers()


if __name__ == "__main__":
    print(f"[Pippit] server jalan di http://0.0.0.0:{PORT}  (POST /tts, POST /avatar ASYNC, GET /avatar/result/<id>, GET /health)", flush=True)
    ThreadingHTTPServer(("0.0.0.0", PORT), H).serve_forever()


### 4) Jalankan server + URL publik gratis (cloudflared)
Biarkan sel ini TERUS berjalan. Muat model pertama kali ~1-2 menit (VoxCPM) + engine avatar.

In [ ]:
import os, subprocess, time, re, urllib.request, http.client

# ====== PILIH MODEL VoxCPM ======
# Free T4 sering OOM dgn VoxCPM2 (apalagi + engine avatar). Kalau crash OOM,
# ganti ke "openbmb/VoxCPM-0.5B" (ringan, ~5GB VRAM).
os.environ["VOXCPM_MODEL"]  = os.environ.get("VOXCPM_MODEL", "openbmb/VoxCPM2")
os.environ["VOXCPM_DEVICE"] = os.environ.get("VOXCPM_DEVICE", "auto")
os.environ["VOXCPM_PORT"]   = "8081"

# ====== CEGAH OOM: pastikan hanya SATU server VoxCPM di GPU ======
# VRAM T4 (~14.5GB) dibagi VoxCPM (~7GB) + SadTalker (~4-5GB). Kalau sel ini
# dijalankan >1x tanpa mematikan server lama, muncul 2 VoxCPM -> GPU penuh ->
# render avatar OOM (HTTP 500). Baris berikut membunuh server & render lama.
subprocess.run("pkill -9 -f pippit_server.py", shell=True)
subprocess.run("pkill -9 -f inference.py", shell=True)
time.sleep(3)
# Kurangi fragmentasi VRAM saat VoxCPM + SadTalker berbagi satu GPU.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# unduh cloudflared (tunnel gratis, tanpa daftar)
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared")
    os.chmod("cloudflared", 0o755)

# jalankan server gabungan di background, log -> server.log
logf = open("server.log", "w")
srv = subprocess.Popen(["python", "-u", "/content/pippit_server.py"],
                       stdout=logf, stderr=subprocess.STDOUT)

print("Memuat model & menunggu server sehat (maks ~12 menit) ...")
ready = False
deadline = time.time() + 12*60
last = 0
while time.time() < deadline:
    if srv.poll() is not None:
        print("\n\u274C Server BERHENTI (crash). Log:\n")
        print(open("server.log").read())
        raise SystemExit("Server gagal start \u2014 lihat traceback di atas.")
    try:
        with open("server.log") as f:
            f.seek(last); chunk = f.read(); last = f.tell()
        if chunk.strip():
            print(chunk, end="")
    except FileNotFoundError:
        pass
    try:
        c = http.client.HTTPConnection("127.0.0.1", 8081, timeout=3)
        c.request("GET", "/health"); r = c.getresponse()
        if r.status == 200:
            ready = True; print("\n\u2705 Server gabungan siap (voice + avatar)."); break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    print("\n\u26A0\uFE0F Server belum siap setelah 12 menit. Log:\n")
    print(open("server.log").read())
    raise SystemExit("Timeout \u2014 coba ganti VOXCPM_MODEL ke openbmb/VoxCPM-0.5B.")

# cek engine avatar
try:
    c = http.client.HTTPConnection("127.0.0.1", 8081, timeout=5)
    c.request("GET", "/avatar/health"); r = c.getresponse()
    print("[avatar/health]", r.read().decode())
except Exception as e:
    print("avatar health check gagal:", e)

# buka tunnel ke port 8081 dan cetak URL (setelah server sehat)
tun = subprocess.Popen(["./cloudflared","tunnel","--url","http://localhost:8081"],
                       stderr=subprocess.PIPE, text=True)
url = None
for line in tun.stderr:
    print(line.strip())
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        print("\n==============================================")
        print(">>> URL SERVER (Voice Clone + Talking Avatar):")
        print(">>>", url)
        print("    - PippitLokal: URL server VoxCPM  = URL di atas")
        print("    - PippitLokal: avatarColabUrl     = URL di atas (SAMA)")
        print("==============================================")
        break

print("\nBiarkan sel ini TERUS BERJALAN. Kalau Colab idle/putus, URL mati (404).")


### Catatan penting
- **Satu URL untuk keduanya**: `/tts` = voice clone, `/avatar` = video talking avatar (SadTalker talk / LivePortrait idle).
- **OOM di T4**: VoxCPM2 + engine avatar berat. Kalau crash, ganti `VOXCPM_MODEL` ke `openbmb/VoxCPM-0.5B` dan/atau turunkan `AVATAR_MAX_SIDE` ke `384`.
- **Timeout**: render avatar butuh menitan — pastikan timeout di app besar (bukan 120s).
- Quick tunnel trycloudflare sementara: kalau Colab idle/putus, URL jadi 404. Jalankan ulang sel runner untuk URL baru, lalu tempel ulang ke PippitLokal.
